This notebook will use the odds API to return odds for the NFL games

In [9]:
import os
import requests
import datetime as date

In [13]:
BASE_URL = "https://api.the-odds-api.com/v4/sports/americanfootball_nfl/odds"

API_KEY = "0052b4345f4731c48e71bd8f621c9c76"

today = date.datetime.today()
today_dow = today.weekday()

print(today)
print(today_dow)

2026-10-28 15:20:00.600928
2


In [ ]:

query_params = {
    "oddsFormat": "american",
    #"sport_key": "americanfootball_nfl",
    "regions": "us",
    "apiKey": API_KEY#,
    #"commenceTimeFrom": "2024-09-01T00:00:00",
    #"commenceTimeTo": "2024-12-31T23:59:59"
    
    
}

response = requests.get(BASE_URL, params=query_params)

print(response.status_code)
print(response.url)


if response.status_code == 200:
    print("Request successful")
    games_data = response.json()
    
else:
    raise ValueError(f"Request failed with status code {response.status_code}")

print(response.json())


In [3]:
games_data = response.json()

print(games_data[0])


{'id': 'ab2865e9f2d4b012de8bc6fb6de11a7c', 'sport_key': 'americanfootball_nfl', 'sport_title': 'NFL', 'commence_time': '2026-10-30T00:15:00Z', 'home_team': 'Green Bay Packers', 'away_team': 'Carolina Panthers', 'bookmakers': [{'key': 'draftkings', 'title': 'DraftKings', 'last_update': '2026-10-28T19:14:20Z', 'markets': [{'key': 'h2h', 'last_update': '2026-10-28T19:14:20Z', 'outcomes': [{'name': 'Carolina Panthers', 'price': 160}, {'name': 'Green Bay Packers', 'price': -192}]}]}, {'key': 'betus', 'title': 'BetUS', 'last_update': '2026-10-28T19:14:22Z', 'markets': [{'key': 'h2h', 'last_update': '2026-10-28T19:14:22Z', 'outcomes': [{'name': 'Carolina Panthers', 'price': 163}, {'name': 'Green Bay Packers', 'price': -190}]}]}, {'key': 'fanduel', 'title': 'FanDuel', 'last_update': '2026-10-28T19:14:23Z', 'markets': [{'key': 'h2h', 'last_update': '2026-10-28T19:14:22Z', 'outcomes': [{'name': 'Carolina Panthers', 'price': 160}, {'name': 'Green Bay Packers', 'price': -190}]}]}, {'key': 'betrive

In [4]:

for sport in games_data:
    print(sport["sport_key"], sport["sport_title"], sport["home_team"], sport["away_team"], sport["bookmakers"])

americanfootball_nfl NFL Green Bay Packers Carolina Panthers [{'key': 'draftkings', 'title': 'DraftKings', 'last_update': '2026-10-28T19:14:20Z', 'markets': [{'key': 'h2h', 'last_update': '2026-10-28T19:14:20Z', 'outcomes': [{'name': 'Carolina Panthers', 'price': 160}, {'name': 'Green Bay Packers', 'price': -192}]}]}, {'key': 'betus', 'title': 'BetUS', 'last_update': '2026-10-28T19:14:22Z', 'markets': [{'key': 'h2h', 'last_update': '2026-10-28T19:14:22Z', 'outcomes': [{'name': 'Carolina Panthers', 'price': 163}, {'name': 'Green Bay Packers', 'price': -190}]}]}, {'key': 'fanduel', 'title': 'FanDuel', 'last_update': '2026-10-28T19:14:23Z', 'markets': [{'key': 'h2h', 'last_update': '2026-10-28T19:14:22Z', 'outcomes': [{'name': 'Carolina Panthers', 'price': 160}, {'name': 'Green Bay Packers', 'price': -190}]}]}, {'key': 'betrivers', 'title': 'BetRivers', 'last_update': '2026-10-28T19:13:54Z', 'markets': [{'key': 'h2h', 'last_update': '2026-10-28T19:13:54Z', 'outcomes': [{'name': 'Carolina 

In [28]:
import pandas as pd

rows = []

for game in games_data:

    team = game["home_team"]
    location = 'home'
    #away_team = game["away_team"]
    commence_time = game["commence_time"]

    for bookmaker in game["bookmakers"]:
        bookmaker_key = bookmaker["key"]
        last_update = bookmaker["last_update"]
        
        if bookmaker["key"] != "draftkings":
            continue

        for market in bookmaker["markets"]:

            if market["key"] != "h2h":
                continue

            outcomes = market["outcomes"]

            # Create dictionary: team -> odds
            odds = {
                outcome["name"]: outcome["price"]
                for outcome in outcomes
            }

            team_odds = odds[team]
            #away_odds = odds[away_team]

            # Negative American odds = favorite
            if team_odds < 0:
                favored = 'favorite'
            else:
                favored = 'underdog'
            rows.append({
                "bookmaker" : bookmaker_key,
                "last_update": last_update,
                "commence_time": commence_time,
                "team": team,
                "location": location,
                "favored": favored
            })

df = pd.DataFrame(rows)

print(df)

     bookmaker           last_update         commence_time  \
0   draftkings  2026-10-28T19:14:20Z  2026-10-30T00:15:00Z   
1   draftkings  2026-10-28T19:14:20Z  2026-11-01T18:00:00Z   
2   draftkings  2026-10-28T19:14:20Z  2026-11-01T18:00:00Z   
3   draftkings  2026-10-28T19:14:20Z  2026-11-01T18:00:00Z   
4   draftkings  2026-10-28T19:14:20Z  2026-11-01T18:00:00Z   
5   draftkings  2026-10-28T19:14:20Z  2026-11-01T18:00:00Z   
6   draftkings  2026-10-28T19:14:20Z  2026-11-01T18:00:00Z   
7   draftkings  2026-10-28T19:14:20Z  2026-11-01T18:00:00Z   
8   draftkings  2026-10-28T19:14:20Z  2026-11-01T18:00:00Z   
9   draftkings  2026-10-28T19:14:20Z  2026-11-01T21:05:00Z   
10  draftkings  2026-10-28T19:14:20Z  2026-11-01T21:25:00Z   
11  draftkings  2026-10-28T19:14:20Z  2026-11-01T21:25:00Z   
12  draftkings  2026-10-28T19:14:20Z  2026-11-02T01:20:00Z   
13  draftkings  2026-10-28T19:14:20Z  2026-11-03T01:20:00Z   
14  draftkings  2026-10-28T19:14:20Z  2026-11-06T01:15:00Z   
15  draf

In [33]:
# get this upcoming sunday date
from datetime import datetime, timedelta

print(df['commence_time'].head())


0    2026-10-30T00:15:00Z
1    2026-11-01T18:00:00Z
2    2026-11-01T18:00:00Z
3    2026-11-01T18:00:00Z
4    2026-11-01T18:00:00Z
Name: commence_time, dtype: object


In [ ]:
df['commence_time'] = pd.to_datetime(df['commence_time']).dt.

,bookmaker,last_update,commence_time,team,location,favored,week_number,NFL_weekly
0,draftkings,2026-10-28T19:14:20Z,2026-10-30T00:15:00Z,Green Bay Packers,home,favorite,2026-10-30,43
1,draftkings,2026-10-28T19:14:20Z,2026-11-01T18:00:00Z,Dallas Cowboys,home,favorite,2026-11-01,44
2,draftkings,2026-10-28T19:14:20Z,2026-11-01T18:00:00Z,Tampa Bay Buccaneers,home,favorite,2026-11-01,44
3,draftkings,2026-10-28T19:14:20Z,2026-11-01T18:00:00Z,Buffalo Bills,home,favorite,2026-11-01,44
4,draftkings,2026-10-28T19:14:20Z,2026-11-01T18:00:00Z,Cincinnati Bengals,home,favorite,2026-11-01,44


In [ ]:



df['NFL_weekly'] = pd.to_datetime(df['commence_time']).dt.strftime('%U')

df.head()




In [16]:
# add team abbreviations to df
team_abbreviations = {
    "Arizona Cardinals": "ARI",
    "Atlanta Falcons": "ATL",
    "Baltimore Ravens": "BAL",
    "Buffalo Bills": "BUF",
    "Carolina Panthers": "CAR",
    "Chicago Bears": "CHI",
    "Cincinnati Bengals": "CIN",
    "Cleveland Browns": "CLE",
    "Dallas Cowboys": "DAL",
    "Denver Broncos": "DEN",
    "Detroit Lions": "DET",
    "Green Bay Packers": "GB",
    "Houston Texans": "HOU",
    "Indianapolis Colts": "IND",
    "Jacksonville Jaguars": "JAX",
    "Kansas City Chiefs": "KC",
    "Las Vegas Raiders": "LV",
    "Los Angeles Chargers": "LAC",
    "Los Angeles Rams": "LAR",
    "Miami Dolphins": "MIA",
    "Minnesota Vikings": "MIN",
    "New England Patriots": "NE",
    "New Orleans Saints": "NO",
    "New York Giants": "NYG",
    "New York Jets": "NYJ",
    "Philadelphia Eagles": "PHI",
    "Pittsburgh Steelers": "PIT",
    "San Francisco 49ers": "SF",
    "Seattle Seahawks": "SEA",
    "Tampa Bay Buccaneers": "TB",
    "Tennessee Titans": "TEN",
    "Washington Commanders": "WAS"
}

df["home_team_abbr"] = df["home_team"].map(team_abbreviations)
df["away_team_abbr"] = df["away_team"].map(team_abbreviations)


C:\Users\onlyu\AppData\Local\Temp\ipykernel_18552\1145455114.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["home_team_abbr"] = df["home_team"].map(team_abbreviations)
C:\Users\onlyu\AppData\Local\Temp\ipykernel_18552\1145455114.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["away_team_abbr"] = df["away_team"].map(team_abbreviations)


In [17]:
df.head()

,week_date,week_number,bookmaker,last_update,commence_time,home_team,away_team,home_odds,away_odds,favored_team,favorite_odds,week_date,week_number,home_team_abbr,away_team_abbr
0,2026-10-18,6,draftkings,2026-10-17T17:36:56Z,2026-10-18T13:30:00Z,Jacksonville Jaguars,Houston Texans,-148,124,Jacksonville Jaguars,-148,2026-10-18,6,JAX,HOU
1,2026-10-18,6,draftkings,2026-10-17T17:36:56Z,2026-10-18T17:00:00Z,Atlanta Falcons,Chicago Bears,100,-120,Chicago Bears,-120,2026-10-18,6,ATL,CHI
2,2026-10-18,6,draftkings,2026-10-17T17:36:56Z,2026-10-18T17:00:00Z,Cleveland Browns,Baltimore Ravens,140,-166,Baltimore Ravens,-166,2026-10-18,6,CLE,BAL
3,2026-10-18,6,draftkings,2026-10-17T17:36:56Z,2026-10-18T17:00:00Z,Philadelphia Eagles,Carolina Panthers,140,-166,Carolina Panthers,-166,2026-10-18,6,PHI,CAR
4,2026-10-18,6,draftkings,2026-10-17T17:36:56Z,2026-10-18T17:00:00Z,Indianapolis Colts,Tennessee Titans,-340,270,Indianapolis Colts,-340,2026-10-18,6,IND,TEN


In [18]:
data_folder = "C:\\Users\\onlyu\\OneDrive\\Coding_Projects\\Fantasy_Football\\data\\vegas_odds"

df.to_csv(f"{data_folder}/nfl_team_odds_{upcoming_sunday.strftime('%Y-%m-%d')}.csv", index=False)

In [ ]:
# load the df to snowflake
import snowflake.connector
import glob

conn = snowflake.connector.connect(
    connection_name="whatever"
)
cursor = conn.cursor()

# test conneciton
cursor.execute("SELECT CURRENT_VERSION()")
print(cursor.fetchone())


In [ ]:

files = glob.glob(r"C:\Users\onlyu\OneDrive\Coding_Projects\Fantasy_Football\data\*nfl_team_odds*.csv")
#
for file_path in files:
    print(file_path)
    cursor.execute(f"PUT file://{file_path} @fantasy_football.raw.ffa_stage OVERWRITE = TRUE")


In [ ]:

# See the staged files
print(cursor.execute("LIST @fantasy_football.raw.ffa_stage;").fetchall())


In [ ]:
cursor.execute("drop table if exists fantasy_football.raw.nfl_team_odds;")

print(cursor.fetchall())

In [ ]:

cursor.execute("""
    CREATE TABLE IF NOT EXISTS fantasy_football.raw.nfl_team_odds (
        date VARCHAR,
        week VARCHAR,	
        home_team VARCHAR,
        away_team VARCHAR,
        home_odds VARCHAR,
        away_odds VARCHAR,
        favored_team VARCHAR,
        favorite_odds VARCHAR,
        home_team_abbr VARCHAR,
        away_team_abbr VARCHAR
    );
""")

print(cursor.fetchall())


In [ ]:

cursor.execute("""
    COPY INTO fantasy_football.raw.nfl_team_odds
    FROM @fantasy_football.raw.ffa_stage
    PATTERN = '.*nfl_team_odds.*'
    FILE_FORMAT = (
        TYPE = CSV 
        FIELD_DELIMITER = ','
        SKIP_HEADER = 1
        )
    ON_ERROR = 'CONTINUE'
""")

print(cursor.fetchall())